### AoC 2025 Day 9

In [1]:
lines = open('input09').readlines()
positions: list[list[int]] = [list(map(int, x.split(','))) for x in lines]
print('num points:', len(positions))
positions[:3]

num points: 496


[[98201, 50069], [98201, 51291], [98234, 51291]]

In [2]:
import plotly.express as px
xplot = [x[0] for x in positions] 
yplot = [x[1] for x in positions] 
fig = px.line(x=xplot, y=yplot, markers=True)
# fig.add_traces(px.line(x=[t[0] for t in biggest_rectangle], 
                    #    y=[t[1] for t in biggest_rectangle], markers=True, color_discrete_sequence=['red']).data)
fig.update_layout(width=600, height=600)
fig.show()

In [3]:
def area(p1: list, p2: list) -> int:
    return (1 + abs(p2[1] - p1[1])) * (1 + abs(p2[0] - p1[0]))

#### Part 1

In [4]:
# try them all
max_area = -1
for i in range(len(positions)):
    for j in range(i+1, len(positions)):
        A = area(positions[i], positions[j])
        max_area = max(max_area, A)
print("Part 1:", max_area)

Part 1: 4733727792


#### Part 2

In [5]:
import dataclasses

@dataclasses.dataclass(frozen=True)
class Point:
    '''an (x,y) point in R2'''
    x: int
    y: int
    
@dataclasses.dataclass(frozen=True)
class Segment:
    '''a line segment between two points. 
    will be either horizontal or vertical in this problem'''
    s1: Point
    s2: Point

In [6]:
# line segments
segments: list[Segment] = [(Segment(Point(*positions[i]), Point(*positions[i+1]))) 
                           for i in range(len(positions)-1)] + \
    [Segment(Point(*positions[-1]), Point(*positions[0]))] 
    # + wrap around to first point in the given list
segments[:3]

[Segment(s1=Point(x=98201, y=50069), s2=Point(x=98201, y=51291)),
 Segment(s1=Point(x=98201, y=51291), s2=Point(x=98234, y=51291)),
 Segment(s1=Point(x=98234, y=51291), s2=Point(x=98234, y=52496))]

In [7]:
max_area = -1
for i in range(len(positions)):
    for j in range(i+1, len(positions)):        
        # opposite corners:
        p1 = Point(*positions[i])
        p2 = Point(*positions[j])
        
        # 4 sides of candidate rectangle
        bottom, top = sorted([p1.y, p2.y])
        left, right = sorted([p1.x, p2.x])
        
        discard = False
        
        # check if any line segments inside rectangle... if so, discard
        for seg in segments:
            if seg.s1.y == seg.s2.y: # horizontal segment
                if bottom < seg.s1.y < top:
                    if (seg.s1.x <= left and seg.s2.x >= right) or \
                       (seg.s2.x <= left and seg.s1.x >= right) or \
                       (left < seg.s1.x < right) or \
                       (left < seg.s2.x < right):
                        discard = True
                        break
                    
            else: # vertical segment
                assert seg.s1.x == seg.s2.x
                if left < seg.s1.x < right: 
                    if (seg.s1.y <= bottom and seg.s2.y >= top) or \
                       (seg.s2.y <= bottom and seg.s1.y >= top) or \
                       (bottom < seg.s1.y < top) or \
                       (bottom < seg.s2.y < top):
                        discard = True
                        break
                               
        if not discard:
            A = area(positions[i], positions[j])
            if A > max_area:
               biggest_rectangle = [positions[i], positions[j]]
               max_area = A
        
print("Part 2:", max_area)

Part 2: 1566346198
